# GP 候選解釋變數的空間前向選擇

本 notebook 使用真實 TCCIP GRID 的 NN-derived GEV parameter estimates，建立 GP mean structure 的候選變數選擇流程。研究問題是：哪些地形、土地覆蓋與海岸距離變數，能改善未見地理區域的參數預測？

土地覆蓋使用 2000 年 reference epoch 的連續面積比例，不使用 0.5 門檻轉成單一類別。

## Block 1：資料與候選變數

三個 response 分別為：

$$
y_\mu(s)=\hat\mu(s),\qquad
y_\sigma(s)=\widehat{\log\sigma}(s),\qquad
y_\xi(s)=\hat\xi(s).
$$

候選變數包括高程、坡度、坡向、地形起伏、崎嶇度、都市／森林／農業／水域比例，以及至最近海岸線的距離。坡向以 `northness + eastness` 成組進入模型，避免角度的 0 度與 360 度不連續問題。

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from spatial_predictor_selection import (
    CANDIDATE_GROUPS,
    load_predictor_selection_data,
    predictor_audit,
    run_all_targets,
)

data = load_predictor_selection_data()
print(f'可用 GRID：{len(data):,}')
pd.DataFrame([
    {'candidate_group': group, 'columns': ' + '.join(columns)}
    for group, columns in CANDIDATE_GROUPS.items()
])

## Block 2：候選變數稽核

先檢查缺值、尺度與相關性。相關性只用來辨識可能的共線性，不直接決定變數是否入選；最終選擇仍以 buffered spatial CV 的 OOF RMSE 為準。特別需要注意 `local_relief_m` 與 `terrain_ruggedness_m` 高度相關，因此不應因兩者都與 response 相關就直接全部加入。

In [ ]:
audit, correlations = predictor_audit(data)
display(audit)
display(correlations.head(15))

## Block 3：固定五區與 buffer

先將 GRID 中心投影為 TWD97 / TM2 公里座標，再以 coordinate-based K-means 建立固定的五個 geographic folds。K-means 的作用只有建立空間集中且互斥的 test regions；它不估計空間自相關距離，也不能自行移除 train-test leakage。

每次保留一區作 test，並移除距離 test GRID 小於 response-specific buffer 的 training GRID：

$$
\mathcal T_b(r_\delta)=\left\{i:\operatorname{fold}(i)\neq b,\ 
\min_{j\in b}d(s_i,s_j)>r_\delta\right\}.
$$

目前沿用的 buffer 為：

$$
r_\mu=55\text{ km},\qquad r_{\log\sigma}=35\text{ km},\qquad r_\xi=30\text{ km}.
$$

## Block 4：Spatial Forward Feature Selection

本階段固定前一階段選出的 kernel，只比較 mean structure。從 intercept-only mean 開始，每一步把尚未入選的候選群組逐一加入，所有模型使用完全相同的 folds、buffer 與 training pool。

$$
g^*=\arg\min_{g\in\mathcal G_{\mathrm{remaining}}}
\operatorname{RMSE}_{\mathrm{buffered\ spatial\ CV}}
(\mathcal S_{k-1}\cup g).
$$

只有當最佳候選使整體 OOF RMSE 至少下降 1% 時才加入：

$$
\frac{\operatorname{RMSE}_{k-1}-\operatorname{RMSE}_{k}}
{\operatorname{RMSE}_{k-1}}>0.01.
$$

1% 是預先設定的 practical-improvement rule，不是顯著水準；用途是避免把固定分割下 0.1% 或 0.2% 的微小波動當成實質改善。

In [ ]:
# 完整 exact-GP FFS 需要數分鐘。若結果 CSV 已存在，預設直接載入。
RUN_FULL_ANALYSIS = False

if RUN_FULL_ANALYSIS:
    trials, selection_path, selected_models = run_all_targets(
        data=data,
        n_folds=5,
        max_train=800,
        min_relative_improvement=0.01,
    )
else:
    table_dir = ROOT / 'results' / 'tables'
    trials = pd.read_csv(table_dir / 'spatial_ffs_trials.csv')
    selection_path = pd.read_csv(table_dir / 'spatial_ffs_selection_path.csv')
    selected_models = pd.read_csv(table_dir / 'spatial_ffs_selected_models.csv')

display(selection_path)

## Block 5：目前的開發階段結果

依 1% stopping rule，目前選出的 mean structures 為：

$$
m_\mu(s)=\beta_0+\beta_1\operatorname{Elevation}(s)
+\beta_2\operatorname{LocalRelief}(s)
+\beta_3p_{\mathrm{agriculture}}(s),
$$

$$
m_{\log\sigma}(s)=\beta_0+\beta_1\operatorname{Elevation}(s)
+\beta_2p_{\mathrm{forest}}(s)
+\beta_3\operatorname{CoastDistance}(s),
$$

$$
m_\xi(s)=\beta_0.
$$

這些結果仍是固定五區下的 model-development evidence，不能直接宣稱為最終無偏預測誤差。

In [ ]:
display(selected_models[[
    'target', 'predictors', 'RMSE', 'MAE', 'Bias', 'kernel', 'nu'
]])

# 顯示未通過 1% 門檻的下一個最佳候選。
selected_steps = selected_models.set_index('target')['step'].to_dict()
stopping_rows = []
for target, selected_step in selected_steps.items():
    next_step = trials.query('target == @target and step == @selected_step + 1')
    if not next_step.empty:
        stopping_rows.append(next_step.sort_values('RMSE').iloc[0])
display(pd.DataFrame(stopping_rows)[[
    'target', 'candidate_group', 'relative_RMSE_improvement', 'raw_p'
]])

## Block 6：如何銜接後續分析

下一階段應固定每個 response 的 selected predictor set，再重新比較 RBF 與 Matérn kernels。接著用選定的三個參數模型建立 mixed OOF pipeline，計算：

$$
RL_T(s)=\mu(s)+\frac{\sigma(s)}{\xi(s)}
\left[\{-\log(1-1/T)\}^{-\xi(s)}-1\right].
$$

最後使用 repeated 或 nested buffered spatial CV 評估 predictor selection、kernel selection、$RL_{50}$ 與 $RL_{100}$，並檢查 OOF residual Moran's $I$ 與 residual variogram。這才是最終可報告的泛化證據。